In [ ]:
import os
from huggingface_hub import login

# Use environment variable for token: export HF_TOKEN="your_token_here"
token = os.getenv("HF_TOKEN")
if token:
    login(token=token)

In [2]:
from transformers import AutoProcessor, AutoModelForCausalLM

model_id = "google/functiongemma-270m-it"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

# JSON Schema formatında fonksiyon tanımı
weather_function = {
    "type": "function",
    "function": {
        "name": "get_current_temperature",
        "description": "Gets the current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, e.g. San Francisco",
                },
            },
            "required": ["location"],
        },
    }
}

# Mesaj formatı - developer rolü ÖNEMLİ!
message = [
    {"role": "developer", "content": "You are a model that can do function calling with the following functions"},
    {"role": "user", "content": "What's the temperature in Istanbul?"}
]

inputs = processor.apply_chat_template(
    message, 
    tools=[weather_function], 
    add_generation_prompt=True, 
    return_dict=True, 
    return_tensors="pt"
)

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
print(output)

<start_function_call>call:get_current_temperature{location:<escape>Istanbul<escape>}<end_function_call>


In [3]:
# Web scraping tools tanımı - JSON Schema formatında
scraping_tools = [
    {
        "type": "function",
        "function": {
            "name": "scrape_webpage",
            "description": "Scrape content from a webpage URL",
            "parameters": {
                "type": "object",
                "properties": {"url": {"type": "string", "description": "The URL to scrape"}},
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_page_title",
            "description": "Get the title of a webpage",
            "parameters": {
                "type": "object",
                "properties": {"url": {"type": "string", "description": "The URL to get title from"}},
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_in_page",
            "description": "Search for text in a webpage",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "The URL to search in"},
                    "query": {"type": "string", "description": "The text to search for"}
                },
                "required": ["url", "query"]
            }
        }
    }
]

queries = [
    "Scrape the content from https://news.ycombinator.com",
    "What is the title of https://github.com",
    "Search for 'Python' in https://wikipedia.org"
]

for query in queries:
    message = [
        {"role": "developer", "content": "You are a model that can do function calling with the following functions"},
        {"role": "user", "content": query}
    ]
    
    inputs = processor.apply_chat_template(
        message, 
        tools=scraping_tools, 
        add_generation_prompt=True, 
        return_dict=True, 
        return_tensors="pt"
    )
    
    out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=64)
    output = processor.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
    print(f"Query: {query}")
    print(f"Response: {output}")
    print("-" * 50)

Query: Scrape the content from https://news.ycombinator.com
Response: <start_function_call>call:scrape_webpage{url:<escape>https://news.ycombinator.com<escape>}<end_function_call>
--------------------------------------------------
Query: What is the title of https://github.com
Response: <start_function_call>call:get_page_title{url:<escape>https://github.com<escape>}<end_function_call>
--------------------------------------------------
Query: Search for 'Python' in https://wikipedia.org
Response: <start_function_call>call:search_in_page{query:<escape>Python<escape>,url:<escape>https://wikipedia.org<escape>}<end_function_call>
--------------------------------------------------
